# FATF-RAG — Reproducible Walkthrough

A retrieval-augmented QA system over the **FATF Recommendations** (the global AML/CFT standard, Feb 2012, 126 pp.).

This notebook reproduces the whole pipeline end to end:
1. Ingest & chunk the PDF
2. Embed chunks and build the FAISS index
3. Retrieve with dense / BM25 / hybrid retrievers
4. Generate a grounded, cited answer (optional — needs Ollama running locally)
5. Evaluate retrieval quantitatively (Recall@k, MRR, Precision@k) + ablation

> **Reproducibility note.** Retrieval and evaluation run fully offline with a local
> sentence-transformers model (downloaded once from Hugging Face). Answer
> *generation* is optional and only runs if a local Llama model is available via Ollama (`ollama pull llama3.2`).

In [ ]:
# Run from the repo root so `import src...` works.
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

## 1. Ingestion & chunking

We extract page text with `pdfplumber`, strip running headers/footers, and split into overlapping, page-tagged chunks. Each chunk records its **page span** and best-effort **section** (e.g. *Recommendation 10*) so every answer can be cited and verified.

In [ ]:
from src.config import Config
from src.ingest import build_chunks

cfg = Config()
chunks = build_chunks(cfg.pdf_path, cfg.chunk_size, cfg.chunk_overlap, cfg.min_chunk_chars)
print(f"{len(chunks)} chunks  |  chunk_size={cfg.chunk_size}, overlap={cfg.chunk_overlap}")
c = chunks[120]
print(f"\nExample chunk #{c.id}  [{c.section}, p.{c.page_start}-{c.page_end}]\n")
print(c.text[:500])

## 2. Embeddings + FAISS index

Local `all-MiniLM-L6-v2` embeddings (384-dim, L2-normalised) indexed with FAISS inner-product search (= cosine). The first run downloads the model (~80 MB).

In [ ]:
from src.embeddings import Embedder
from src.vectorstore import VectorStore

embedder = Embedder(cfg.embed_model)
embs = embedder.encode([c.text for c in chunks])
store = VectorStore(chunks, embs)
store.save(cfg.index_path, cfg.chunks_path)   # persist for the CLI / web app
print("indexed", store.ntotal, "chunks, dim", store.dim)

## 3. Retrieval: dense vs BM25 vs hybrid

- **dense** — semantic similarity; good for paraphrased / conceptual questions.
- **bm25** — lexical; good for exact terms, numbers, Recommendation IDs.
- **hybrid** — min-max normalise both score lists, combine as `alpha*dense + (1-alpha)*bm25`.

Compare what each retriever surfaces for the same query.

In [ ]:
from src.retriever import Retriever

q = "What are the wire transfer requirements under Recommendation 16?"
for mode in ["dense", "bm25", "hybrid"]:
    r = Retriever(store, embedder, mode=mode, hybrid_alpha=cfg.hybrid_alpha)
    top = r.retrieve(q, top_k=3)
    print(f"\n=== {mode} ===")
    for s in top:
        print(f"  {s.citation}  score={s.score:.3f}")

## 4. Grounded answer (optional generation)

The pipeline retrieves passages and asks the LLM to answer **using only that context**, citing the Recommendation + page. Without Ollama running it returns the passages (retrieval-only mode) — the pipeline never hard-fails.

In [ ]:
from src.pipeline import RAGPipeline

pipe = RAGPipeline(cfg, store=store, embedder=embedder)
resp = pipe.query("When must enhanced due diligence be applied?", generate=True)
print("LLM available:", pipe.llm.available, "\n")
print(resp.answer)
print("\n--- sources ---")
print(resp.format_sources())

## 5. Retrieval evaluation

We hand-built a 15-question test set (`eval/testset.json`) with **gold pages** located manually in the source. A retrieved chunk is *relevant* if its page span intersects the gold pages. We report **Recall@k** (= hit rate here), **MRR**, and **Precision@k**.

In [ ]:
from eval.evaluate import evaluate_retriever, load_testset
import pandas as pd

ts = load_testset()["questions"]
rows = []
for mode in ["bm25", "dense", "hybrid"]:
    r = Retriever(store, embedder, mode=mode, hybrid_alpha=cfg.hybrid_alpha)
    res = evaluate_retriever(r, ts, k=5)
    rows.append({"retriever": mode, "Recall@5": res["recall@k"],
                 "MRR": res["mrr"], "Precision@5": res["precision@k"]})
pd.DataFrame(rows).set_index("retriever")

### Ablation: chunk size

Smaller chunks raise precision (less off-topic text) but can split an answer; larger chunks improve recall but dilute. Run the full ablation from the CLI:

```bash
python -m eval.evaluate --ablation
```

In [ ]:
# Quick chunk-size sweep (rebuilds the index per setting; ~1-2 min)
for cs in [600, 900, 1200]:
    c2 = Config(); c2.chunk_size = cs
    ch = build_chunks(c2.pdf_path, c2.chunk_size, c2.chunk_overlap, c2.min_chunk_chars)
    st = VectorStore(ch, embedder.encode([x.text for x in ch]))
    res = evaluate_retriever(Retriever(st, embedder, mode="hybrid"), ts, k=5)
    print(f"size={cs:4d} ({len(ch):3d} chunks)  Recall@5={res['recall@k']}  MRR={res['mrr']}  P@5={res['precision@k']}")

## Takeaways

- The corpus is small and highly structured, so **exact-match (BM25) is a very strong baseline** — it already hits Recall@5 = 1.0 on our test set. The value of dense/hybrid shows up on **paraphrased** questions that don't share surface terms with the text.
- **Page-tagged chunks** make every answer auditable against the source PDF — essential for a legal/compliance use case.
- The honest limitation: our gold labels are page-level and hand-built by a small team, so the test set is a *sanity check*, not a benchmark. See the README for the full discussion.